# Sequential Fine-tuning on Skywork (Colab Version)

This notebook runs sequential fine-tuning of Skywork-Reward-V2-Qwen3-0.6B on:
- Stage 1: Universal Turing Machine (UTM) data
- Stage 2: Context-Tree Weighting (CTW) data

**Memory optimized for Google Colab (15GB GPU)**

⚠️ **Important**: Enable GPU runtime (Runtime → Change runtime type → GPU)

## 1. Setup and Installation

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Clone repository
!git clone https://github.com/google-deepmind/neural_networks_solomonoff_induction.git
%cd neural_networks_solomonoff_induction

In [ ]:
# Install dependencies (lightweight for Colab)
!pip install -q torch transformers numpy jax dm-haiku optax accelerate bitsandbytes

## 2. Memory-Efficient Configuration

In [ ]:
import torch
import gc

# Clear cache
torch.cuda.empty_cache()
gc.collect()

# Memory-efficient settings for Colab
CONFIG = {
    'model_name': 'Skywork/Skywork-Reward-V2-Qwen3-0.6B',
    'batch_size': 4,           # Reduced from 32
    'seq_length': 128,         # Reduced from 256
    'utm_steps': 2000,         # Reduced from 10000
    'ctw_steps': 1000,         # Reduced from 5000
    'utm_lr': 5e-5,
    'ctw_lr': 2e-5,
    'log_every': 25,
    'save_every': 500,
    'output_dir': './checkpoints/skywork_colab',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'gradient_checkpointing': True,  # Save memory
    'use_8bit': True,                # 8-bit quantization
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

print(f"\nGPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Create Lightweight Training Script

In [ ]:
%%writefile colab_train.py
#!/usr/bin/env python3
"""Colab-optimized sequential fine-tuning."""

import argparse
import torch
import torch.nn.functional as F
import numpy as np
import gc
from pathlib import Path

# Memory optimization
torch.backends.cudnn.benchmark = True


def load_model_8bit(model_name, device):
    """Load model with 8-bit quantization for memory efficiency."""
    try:
        from transformers import AutoModelForCausalLM, BitsAndBytesConfig
        
        quantization_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_threshold=6.0,
        )
        
        print(f"Loading {model_name} with 8-bit quantization...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quantization_config,
            device_map='auto',
            trust_remote_code=True,
        )
        
        # Enable gradient checkpointing
        model.gradient_checkpointing_enable()
        
        return model
    except Exception as e:
        print(f"8-bit loading failed: {e}")
        print("Falling back to standard loading...")
        from transformers import AutoModelForCausalLM
        return AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map='auto',
            trust_remote_code=True,
        )


def train_step(model, batch, optimizer, device):
    """Single training step with memory cleanup."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    
    x = torch.from_numpy(batch).long().to(device)
    
    outputs = model(x, labels=x)
    loss = outputs.loss
    
    loss.backward()
    optimizer.step()
    
    loss_val = float(loss.detach().cpu().item())
    
    # Memory cleanup
    del outputs, loss, x
    torch.cuda.empty_cache()
    
    return loss_val


def main(args):
    print("="*60)
    print("Colab Sequential Fine-tuning")
    print("="*60)
    
    # Load model
    model = load_model_8bit(args.model_name, args.device)
    
    # Prepare optimizer (only trainable params)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=args.lr)
    
    print(f"\nTrainable parameters: {sum(p.numel() for p in trainable_params):,}")
    print(f"Total steps: {args.steps}")
    print(f"Batch size: {args.batch_size}")
    print(f"Sequence length: {args.seq_length}\n")
    
    # Simple training loop with dummy data (replace with real data generator)
    for step in range(args.steps):
        # Generate dummy batch (replace with actual data generator)
        batch = np.random.randint(0, 1000, (args.batch_size, args.seq_length))
        
        loss = train_step(model, batch, optimizer, args.device)
        
        if step % args.log_every == 0:
            print(f"Step {step}/{args.steps} | Loss: {loss:.4f}")
        
        if step % args.save_every == 0 and step > 0:
            checkpoint_path = Path(args.output_dir) / f"checkpoint_step_{step}.pt"
            checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss,
            }, checkpoint_path)
            print(f"Saved checkpoint to {checkpoint_path}")
    
    print("\nTraining complete!")


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--model_name', type=str, required=True)
    parser.add_argument('--steps', type=int, default=1000)
    parser.add_argument('--batch_size', type=int, default=4)
    parser.add_argument('--seq_length', type=int, default=128)
    parser.add_argument('--lr', type=float, default=5e-5)
    parser.add_argument('--log_every', type=int, default=25)
    parser.add_argument('--save_every', type=int, default=500)
    parser.add_argument('--output_dir', type=str, default='./checkpoints')
    parser.add_argument('--device', type=str, default='cuda')
    args = parser.parse_args()
    main(args)

## 4. Run Stage 1: UTM Fine-tuning (Memory-Efficient)

In [ ]:
# Clear memory before starting
torch.cuda.empty_cache()
gc.collect()

# Run UTM stage
!python colab_train.py \
  --model_name "{CONFIG['model_name']}" \
  --steps {CONFIG['utm_steps']} \
  --batch_size {CONFIG['batch_size']} \
  --seq_length {CONFIG['seq_length']} \
  --lr {CONFIG['utm_lr']} \
  --log_every {CONFIG['log_every']} \
  --save_every {CONFIG['save_every']} \
  --output_dir "{CONFIG['output_dir']}/stage1_utm" \
  --device {CONFIG['device']}

## 5. Monitor GPU Memory Usage

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"Max allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
else:
    print("CUDA not available")

## 6. Full Pipeline with Real Data (If Memory Allows)

In [ ]:
# Try running the full pipeline with ultra-low memory settings
!python torch_training/train_sequential_finetune.py \
  --model_name_or_path "{CONFIG['model_name']}" \
  --use_hf \
  --stage utm \
  --utm_steps {CONFIG['utm_steps']} \
  --utm_lr {CONFIG['utm_lr']} \
  --batch_size {CONFIG['batch_size']} \
  --seq_length {CONFIG['seq_length']} \
  --memory_size 5 \
  --maximum_steps 50 \
  --maximum_program_length 50 \
  --save_every {CONFIG['save_every']} \
  --log_every {CONFIG['log_every']} \
  --output_dir "{CONFIG['output_dir']}" \
  --device {CONFIG['device']}

## 7. Emergency: Ultra Low Memory Mode

If still running out of memory, try these:

In [ ]:
# ULTRA LOW MEMORY SETTINGS
ULTRA_LOW_CONFIG = {
    'batch_size': 1,
    'seq_length': 64,
    'utm_steps': 500,
    'ctw_steps': 250,
}

!python torch_training/train_sequential_finetune.py \
  --model_name_or_path "{CONFIG['model_name']}" \
  --use_hf \
  --stage all \
  --utm_steps {ULTRA_LOW_CONFIG['utm_steps']} \
  --ctw_steps {ULTRA_LOW_CONFIG['ctw_steps']} \
  --batch_size {ULTRA_LOW_CONFIG['batch_size']} \
  --seq_length {ULTRA_LOW_CONFIG['seq_length']} \
  --log_every 10 \
  --save_every 100 \
  --output_dir "{CONFIG['output_dir']}_ultra_low" \
  --device {CONFIG['device']}

## 8. Download Checkpoints to Local Machine

In [ ]:
# Zip checkpoints
!zip -r checkpoints.zip {CONFIG['output_dir']}

# Download
from google.colab import files
files.download('checkpoints.zip')

## 9. Alternative: Use Smaller Model

In [ ]:
# Train custom LSTM instead (much more memory efficient)
!python torch_training/train_sequential_finetune.py \
  --architecture lstm \
  --hidden_dim 128 \
  --num_layers 2 \
  --embedding_dim 64 \
  --stage all \
  --utm_steps 5000 \
  --ctw_steps 2500 \
  --batch_size 16 \
  --seq_length 256 \
  --output_dir "./checkpoints/lstm_colab" \
  --device {CONFIG['device']}

print("\n✅ LSTM training is much more memory efficient!")

## Troubleshooting

### Out of Memory
1. Reduce `batch_size` to 1
2. Reduce `seq_length` to 64
3. Use LSTM instead of Skywork
4. Enable Colab Pro for more GPU memory

### Slow Training
1. Reduce `utm_steps` and `ctw_steps`
2. Increase `batch_size` if memory allows
3. Use gradient accumulation

### Model Download Fails
1. Check internet connection
2. Login to HuggingFace: `!huggingface-cli login`
3. Try a different model variant